# LeetCode 3594 — Minimum Time to Transport All Individuals bằng Python

- Link: https://leetcode.com/problems/minimum-time-to-transport-all-individuals/
- Độ khó: **Hard**
- Chủ đề: **Bitmask**, **duyệt tập con (submask)**, **đồ thị trạng thái có chu trình**, **Dijkstra** (`heapq`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minTime(self, n: int, k: int, m: int, time: List[int], mul: List[float]) -> float:
```

## 1. Mục tiêu học

1. Mô hình hóa bài "qua sông" thành **đồ thị trạng thái**: (ai còn ở bờ xuất phát, giai đoạn hiện tại, thuyền ở bờ nào).
2. Hiểu vì sao **quy hoạch động thông thường không dùng được** (trạng thái có chu trình: người đi rồi lại quay về) nhưng **Dijkstra** dùng được (mọi trọng số dương).
3. Thành thạo kỹ thuật **duyệt mọi tập con của một mask**: `sub = (sub - 1) & mask`.
4. Tiền xử lý `max_time[mask]` bằng bit thấp nhất để tính thời gian chuyến đi trong `O(1)`.
5. Kiểm chứng bằng vét cạn đường đi và bằng Bellman-Ford trên dữ liệu nhỏ; ước lượng thời gian ở giới hạn `n = 12`.

## 2. Đề bài & ví dụ

Có `n` người ở trại, cần qua sông bằng **một thuyền** chở tối đa `k` người. Điều kiện môi trường thay đổi theo chu kỳ `m` **giai đoạn**; giai đoạn `j` có hệ số tốc độ `mul[j]`. Người `i` chèo một mình mất `time[i]` phút trong điều kiện bình thường.

- Một nhóm `g` xuất phát ở giai đoạn `j` qua sông mất `d = max(time[i] với i ∈ g) × mul[j]` phút; sau đó giai đoạn tăng thêm `floor(d) % m` (vòng tròn theo `m`).
- Nếu còn người ở trại, **một người** ở bờ đích phải chèo thuyền về: người `r` mất `time[r] × mul[giai đoạn hiện tại]`, giai đoạn tăng thêm `floor(thời gian về) % m`.

Trả về **tổng thời gian nhỏ nhất** để đưa mọi người sang bờ đích, hoặc `-1` nếu không thể.

**Ví dụ 1:** `n=1, k=1, m=2, time=[5], mul=[1.0,1.3]` → `5.0`.

**Ví dụ 2:** `n=3, k=2, m=3, time=[2,5,8], mul=[1.0,1.5,0.75]` → `14.5`: đưa {0,2} sang (8, giai đoạn 0→2), người 0 về (2×0.75=1.5, giai đoạn 2→0), đưa {0,1} sang (5). Tổng 14.5.

**Ví dụ 3:** `n=2, k=1, m=2, time=[10,10], mul=[2.0,2.0]` → `-1` (thuyền chỉ chở 1 người, ai sang cũng phải quay về).

**Ràng buộc:** `1 ≤ n ≤ 12`, `1 ≤ k ≤ 5`, `1 ≤ m ≤ 5`, `1 ≤ time[i] ≤ 100`, `0.5 ≤ mul[i] ≤ 2.0`.

In [1]:
from typing import List
from math import floor, isclose
import heapq

examples = [
    (1, 1, 2, [5], [1.0, 1.3], 5.0),
    (3, 2, 3, [2, 5, 8], [1.0, 1.5, 0.75], 14.5),
    (2, 1, 2, [10, 10], [2.0, 2.0], -1.0),
]
for n, k, m, t, mul, exp in examples:
    print(f"n={n}, k={k}, m={m}, time={t}, mul={mul} -> kỳ vọng {exp}")

n=1, k=1, m=2, time=[5], mul=[1.0, 1.3] -> kỳ vọng 5.0
n=3, k=2, m=3, time=[2, 5, 8], mul=[1.0, 1.5, 0.75] -> kỳ vọng 14.5
n=2, k=1, m=2, time=[10, 10], mul=[2.0, 2.0] -> kỳ vọng -1.0


## 3. Kiến thức nền

### 3.1. Biểu diễn tập người bằng bitmask

Với `n ≤ 12`, tập người ở trại là một số nguyên `mask` trong `[0, 2^n)`: bit `i` bằng 1 nếu người `i` còn ở trại. Các phép thường dùng:

- `mask >> i & 1`: người `i` có trong tập không.
- `mask ^ sub`: bỏ tập `sub` (đã là tập con) khỏi `mask`.
- `mask | (1 << r)`: thêm người `r`.
- `mask & -mask`: bit thấp nhất; `.bit_length() - 1` cho chỉ số của nó.

In [2]:
mask = 0b1011          # người 0, 1, 3 ở trại
print([i for i in range(4) if mask >> i & 1])
lb = mask & -mask
print('bit thấp nhất:', bin(lb), '-> người', lb.bit_length() - 1)
print('bỏ {0,3}:', bin(mask ^ 0b1001), '| thêm người 2:', bin(mask | 1 << 2))

[0, 1, 3]
bit thấp nhất: 0b1 -> người 0
bỏ {0,3}: 0b10 | thêm người 2: 0b1111


### 3.2. Duyệt mọi tập con khác rỗng của `mask`

Mẹo kinh điển: `sub = mask; while sub: ...; sub = (sub - 1) & mask`. Phép `(sub - 1) & mask` cho tập con **kế tiếp nhỏ hơn**. Tổng số cặp (mask, sub) trên mọi mask là `3^n`.

In [3]:
mask = 0b1011
sub = mask
subs = []
while sub:
    subs.append(bin(sub))
    sub = (sub - 1) & mask
print(len(subs), subs)
print('Tổng số cặp (mask, sub) khi n = 12:', 3 ** 12)

7 ['0b1011', '0b1010', '0b1001', '0b1000', '0b11', '0b10', '0b1']
Tổng số cặp (mask, sub) khi n = 12: 531441


### 3.3. Tiền xử lý `max_time[mask]`

`max_time[s] = max(max_time[s bỏ bit thấp nhất], time[chỉ số bit thấp nhất])` — mỗi mask tính trong `O(1)`. Tương tự cho `popcount[s]`.

In [4]:
def precompute(time):
    n = len(time)
    mx = [0] * (1 << n)
    pc = [0] * (1 << n)
    for s in range(1, 1 << n):
        lb = s & -s
        mx[s] = max(mx[s ^ lb], time[lb.bit_length() - 1])
        pc[s] = pc[s ^ lb] + 1
    return mx, pc

mx, pc = precompute([2, 5, 8])
for s in range(8):
    print(format(s, '03b'), 'max =', mx[s], 'số người =', pc[s])

000 max = 0 số người = 0
001 max = 2 số người = 1
010 max = 5 số người = 1
011 max = 5 số người = 2
100 max = 8 số người = 1
101 max = 8 số người = 2
110 max = 8 số người = 2
111 max = 8 số người = 3


### 3.4. `heapq` và Dijkstra

Dijkstra tìm đường ngắn nhất từ một đỉnh nguồn trên đồ thị **trọng số không âm**: luôn lấy đỉnh có khoảng cách tạm nhỏ nhất ra khỏi heap; khi một đỉnh được lấy ra lần đầu, khoảng cách của nó đã tối ưu. Với Python dùng `heapq.heappush/heappop` và "xóa lười": bỏ qua phần tử heap đã lỗi thời (`d > dist[u]`).

In [5]:
def dijkstra_demo(adj, src):
    dist = {src: 0}
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in adj.get(u, []):
            if d + w < dist.get(v, float('inf')):
                dist[v] = d + w
                heapq.heappush(pq, (d + w, v))
    return dist

adj = {'A': [('B', 4), ('C', 1)], 'C': [('B', 2), ('D', 7)], 'B': [('D', 1)]}
print(dijkstra_demo(adj, 'A'))

{'A': 0, 'B': 3, 'C': 1, 'D': 4}


### 3.5. `floor` với số thực

Giai đoạn tăng `floor(d) % m`. Với số thực, `floor(7.9) = 7`. Lưu ý dấu phẩy động: `100 * 0.57` trong máy tính là `56.99999999999999` nên `floor` cho `56`. Lời giải tham khảo của LeetCode cũng dùng phép nhân số thực như vậy nên ta làm **y hệt** (nhân trực tiếp `time × mul`) để khớp kết quả chấm.

In [6]:
print(floor(7.9), floor(8.0), 100 * 0.57, floor(100 * 0.57))
st, m = 2, 3
d = 2 * 0.75
print('thời gian =', d, '-> giai đoạn mới =', (st + floor(d) % m) % m)

7 8 56.99999999999999 56
thời gian = 1.5 -> giai đoạn mới = 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Trạng thái đầy đủ của "thế giới" tại một thời điểm gồm:

1. Ai còn ở trại (`mask`).
2. Giai đoạn hiện tại `stage ∈ [0, m)`.
3. Thuyền đang ở bờ nào (`side`: 0 = trại, 1 = đích).

Tổng thời gian đã trôi qua **không** ảnh hưởng đến tương lai (giai đoạn đã gói gọn thông tin cần), nên đây là bài **đường đi ngắn nhất**: nguồn `(full, 0, 0)`, đích là mọi trạng thái `(0, *, 1)`.

Tính tay ví dụ 2 (`time=[2,5,8]`, `mul=[1.0,1.5,0.75]`, `k=2`):

| bước | hành động | thời gian | giai đoạn |
|---|---|---|---|
| 0 | bắt đầu `mask=111` | 0 | 0 |
| 1 | {0,2} sang: `8×1.0` | 8.0 | (0 + 8%3) % 3 = 2 |
| 2 | 0 về: `2×0.75` | 1.5 | (2 + 1%3) % 3 = 0 |
| 3 | {0,1} sang: `5×1.0` | 5.0 | 2 |

Tổng `14.5`. Chú ý: người 0 đi **hai lần** — trạng thái có thể lặp, đồ thị có **chu trình** ⇒ không có thứ tự topo để làm DP thẳng.

**Khi nào `-1`?** Nếu `k = 1` và `n ≥ 2`: mỗi chuyến chỉ một người sang, người đó lại phải về ⇒ không bao giờ xong. Nếu `k ≥ 2` luôn làm được (đưa 2 người, 1 người về: mỗi vòng giảm 1).

In [7]:
def simulate(time, mul, m, plan):
    """plan: danh sách ('go', [người]) hoặc ('back', người). Trả về tổng thời gian."""
    st, total = 0, 0.0
    for kind, who in plan:
        if kind == 'go':
            d = max(time[i] for i in who) * mul[st]
        else:
            d = time[who] * mul[st]
        total += d
        new_st = (st + floor(d) % m) % m
        print(f"{kind:>4} {who}: {d:.2f} phút, giai đoạn {st} -> {new_st}, tổng {total:.2f}")
        st = new_st
    return total

simulate([2, 5, 8], [1.0, 1.5, 0.75], 3, [('go', [0, 2]), ('back', 0), ('go', [0, 1])])

  go [0, 2]: 8.00 phút, giai đoạn 0 -> 2, tổng 8.00
back 0: 1.50 phút, giai đoạn 2 -> 0, tổng 9.50
  go [0, 1]: 5.00 phút, giai đoạn 0 -> 2, tổng 14.50


14.5

## 5. Cách 1 — Brute Force: vét cạn mọi chuỗi chuyến đi (không lặp trạng thái)

Đường đi ngắn nhất (trọng số dương) không bao giờ đi qua một trạng thái hai lần. Vậy ta có thể DFS mọi đường đi **đơn** (không lặp trạng thái) từ nguồn, cập nhật đáp án khi đến trạng thái kết thúc. Số đường đơn bùng nổ rất nhanh ⇒ chỉ dùng với `n ≤ 3` làm oracle.

In [8]:
def min_time_brute(n, k, m, time, mul):
    full = (1 << n) - 1
    best = [float('inf')]
    on_path = set()

    def dfs(mask, st, side, total):
        if total >= best[0]:
            return
        if side == 1 and mask == 0:
            best[0] = total
            return
        key = (mask, st, side)
        if key in on_path:
            return
        on_path.add(key)
        if side == 0:
            for sub in range(1, full + 1):
                if sub & mask == sub and bin(sub).count('1') <= k:
                    d = max(time[i] for i in range(n) if sub >> i & 1) * mul[st]
                    dfs(mask ^ sub, (st + floor(d) % m) % m, 1, total + d)
        else:
            for r in range(n):
                if not mask >> r & 1:
                    d = time[r] * mul[st]
                    dfs(mask | 1 << r, (st + floor(d) % m) % m, 0, total + d)
        on_path.discard(key)

    dfs(full, 0, 0, 0.0)
    return best[0] if best[0] < float('inf') else -1.0

for n, k, m, t, mul, exp in examples:
    print(min_time_brute(n, k, m, t, mul), '| kỳ vọng', exp)

5.0 | kỳ vọng 5.0
14.5 | kỳ vọng 14.5
-1.0 | kỳ vọng -1.0


**Độ phức tạp:** số đường đơn có thể mũ theo số trạng thái — rất chậm. Cắt nhánh `total ≥ best` giúp phần nào nhưng không đổi bản chất.

**Nút thắt:** mỗi trạng thái bị thăm lại rất nhiều lần qua các đường khác nhau. Cần thuật toán đường ngắn nhất "chuẩn" trên đồ thị trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đồ thị trạng thái:** đỉnh = `(mask, stage, side)`, tối đa `2^12 × 5 × 2 = 40 960` đỉnh.

- Từ `(mask, st, 0)`: với mỗi tập con `sub ⊆ mask`, `1 ≤ |sub| ≤ k`, cạnh tới `(mask ^ sub, st', 1)` trọng số `d = max_time[sub] × mul[st]`.
- Từ `(mask, st, 1)` với `mask ≠ 0`: với mỗi người `r` ở bờ đích, cạnh tới `(mask | 1<<r, st', 0)` trọng số `time[r] × mul[st]`.

**Insight 2 — trọng số dương ⇒ Dijkstra.** `time ≥ 1`, `mul ≥ 0.5` nên mọi cạnh `≥ 0.5 > 0`. Dù có chu trình, Dijkstra vẫn đúng.

**Insight 3 — dừng sớm.** Trạng thái kết thúc `(0, *, 1)` được lấy ra khỏi heap lần đầu ⇒ đó là đáp án; không cần thăm hết đồ thị.

Quy tắc ngắn:

- **Nếu** `side = 1` và `mask = 0` khi pop ⇒ trả về `d`.
- **Nếu** heap rỗng mà chưa tới đích ⇒ `-1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bellman-Ford (lặp nới lỏng đến khi ổn định)

Liệt kê toàn bộ cạnh, lặp "nới lỏng" mọi cạnh cho tới khi không khoảng cách nào giảm. Đúng với đồ thị không có chu trình âm, nhưng mỗi vòng duyệt hết cạnh và có thể cần nhiều vòng.

In [9]:
def build_edges(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    edges = []
    for mask in range(full + 1):
        for st in range(m):
            sub = mask
            while sub:
                if pc[sub] <= k:
                    d = mx[sub] * mul[st]
                    edges.append(((mask, st, 0), (mask ^ sub, (st + floor(d) % m) % m, 1), d))
                sub = (sub - 1) & mask
            dest = full ^ mask
            if mask:                      # còn người ở trại thì mới phải quay về
                for r in range(n):
                    if dest >> r & 1:
                        d = time[r] * mul[st]
                        edges.append(((mask, st, 1), (mask | 1 << r, (st + floor(d) % m) % m, 0), d))
    return edges

def min_time_bellman(n, k, m, time, mul):
    full = (1 << n) - 1
    edges = build_edges(n, k, m, time, mul)
    dist = {(full, 0, 0): 0.0}
    changed = True
    rounds = 0
    while changed:
        changed = False
        rounds += 1
        for u, v, w in edges:
            if u in dist and dist[u] + w < dist.get(v, float('inf')) - 1e-12:
                dist[v] = dist[u] + w
                changed = True
    ans = min((dist[(0, st, 1)] for st in range(m) if (0, st, 1) in dist), default=-1.0)
    return ans, rounds, len(edges)

for n, k, m, t, mul, exp in examples:
    print(min_time_bellman(n, k, m, t, mul), '| kỳ vọng', exp)

(5.0, 2, 2) | kỳ vọng 5.0
(14.5, 3, 81) | kỳ vọng 14.5
(-1.0, 2, 12) | kỳ vọng -1.0


**Độ phức tạp:** `O(V · E)` trong trường hợp xấu (`V` đỉnh, `E` cạnh); thực tế số vòng nhỏ nhưng mỗi vòng duyệt hàng trăm nghìn cạnh — quá chậm với `n = 12`.

### 7.2. Cách 3 — Dijkstra gộp "đi + về" thành một bước

Bỏ chiều `side`: trạng thái `(mask, st)` luôn là "thuyền ở trại". Một bước = chọn nhóm đi sang **và** người quay về (nếu còn người). Ít đỉnh hơn nhưng mỗi đỉnh có `#sub × #r` cạnh.

In [10]:
def min_time_merged(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    INF = float('inf')
    dist = {(full, 0): 0.0}
    pq = [(0.0, full, 0)]
    best = INF
    while pq:
        d, mask, st = heapq.heappop(pq)
        if d > dist[(mask, st)] or d >= best:
            continue
        sub = mask
        while sub:
            if pc[sub] <= k:
                t1 = mx[sub] * mul[st]
                st1 = (st + floor(t1) % m) % m
                rest = mask ^ sub
                if rest == 0:
                    best = min(best, d + t1)
                else:
                    dest = full ^ rest
                    for r in range(n):
                        if dest >> r & 1:
                            t2 = time[r] * mul[st1]
                            st2 = (st1 + floor(t2) % m) % m
                            key = (rest | 1 << r, st2)
                            nd = d + t1 + t2
                            if nd < dist.get(key, INF):
                                dist[key] = nd
                                heapq.heappush(pq, (nd, rest | 1 << r, st2))
            sub = (sub - 1) & mask
    return best if best < INF else -1.0

for n, k, m, t, mul, exp in examples:
    print(min_time_merged(n, k, m, t, mul), '| kỳ vọng', exp)

5.0 | kỳ vọng 5.0
14.5 | kỳ vọng 14.5
-1.0 | kỳ vọng -1.0


Chú ý: vì kết thúc nằm "giữa" một bước (sau chuyến đi, không có chuyến về), ta không thể dừng ngay khi pop; thay vào đó giữ `best` và **cắt** các trạng thái có `d ≥ best`.

**Độ phức tạp:** `O(m · 3^n · n · log)` trong trường hợp xấu.

### 7.3. Cách 4 — Dijkstra hai phía (lời giải chính)

Tách mỗi bước thành hai cạnh (đi / về), thêm chiều `side`. Số cạnh giảm từ `#sub × #r` xuống `#sub + #r` mỗi trạng thái, và đích `(0, *, 1)` là một đỉnh thật ⇒ dừng ngay khi pop.

In [11]:
def min_time_dijkstra(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    INF = float('inf')
    dist = {(full, 0, 0): 0.0}
    pq = [(0.0, full, 0, 0)]
    while pq:
        d, mask, st, side = heapq.heappop(pq)
        if d > dist[(mask, st, side)]:
            continue
        if side == 1 and mask == 0:
            return d                              # lần đầu chạm đích = tối ưu
        if side == 0:
            sub = mask
            while sub:
                if pc[sub] <= k:
                    t = mx[sub] * mul[st]
                    key = (mask ^ sub, (st + floor(t) % m) % m, 1)
                    if d + t < dist.get(key, INF):
                        dist[key] = d + t
                        heapq.heappush(pq, (d + t, *key))
                sub = (sub - 1) & mask
        else:
            dest = full ^ mask
            while dest:
                lb = dest & -dest
                dest ^= lb
                t = time[lb.bit_length() - 1] * mul[st]
                key = (mask | lb, (st + floor(t) % m) % m, 0)
                if d + t < dist.get(key, INF):
                    dist[key] = d + t
                    heapq.heappush(pq, (d + t, *key))
    return -1.0

for n, k, m, t, mul, exp in examples:
    print(min_time_dijkstra(n, k, m, t, mul), '| kỳ vọng', exp)

5.0 | kỳ vọng 5.0
14.5 | kỳ vọng 14.5
-1.0 | kỳ vọng -1.0


**Độ phức tạp:** số đỉnh `V = 2 · m · 2^n`, số cạnh `E ≤ m · (Σ_mask #sub(≤k) + n · 2^n)`; Dijkstra `O(E log V)`. Với `n = 12, k = 5, m = 5` khoảng vài triệu thao tác trong trường hợp xấu, thực tế dừng sớm.

## 8. TRACE từng bước

Trace Dijkstra hai phía trên ví dụ 2: in từng đỉnh được pop (theo thứ tự khoảng cách tăng dần) cùng số đỉnh mới được cập nhật.

In [12]:
def trace_dijkstra(n, k, m, time, mul, limit=25):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    INF = float('inf')
    dist = {(full, 0, 0): 0.0}
    pq = [(0.0, full, 0, 0)]
    step = 0
    while pq:
        d, mask, st, side = heapq.heappop(pq)
        if d > dist[(mask, st, side)]:
            continue
        step += 1
        where = 'trại' if side == 0 else 'đích'
        print(f"#{step:>2} pop d={d:6.2f} | ở trại={format(mask, f'0{n}b')} giai đoạn={st} thuyền ở {where}")
        if side == 1 and mask == 0:
            print('==> Đã đưa hết mọi người, đáp án =', d)
            return d
        if step >= limit:
            print('... (dừng in)')
            break
        updates = []
        if side == 0:
            sub = mask
            while sub:
                if pc[sub] <= k:
                    t = mx[sub] * mul[st]
                    key = (mask ^ sub, (st + floor(t) % m) % m, 1)
                    if d + t < dist.get(key, INF):
                        dist[key] = d + t
                        heapq.heappush(pq, (d + t, *key))
                        updates.append(f"đi {format(sub, f'0{n}b')}:{t:.2f}")
                sub = (sub - 1) & mask
        else:
            for r in range(n):
                if not mask >> r & 1:
                    t = time[r] * mul[st]
                    key = (mask | 1 << r, (st + floor(t) % m) % m, 0)
                    if d + t < dist.get(key, INF):
                        dist[key] = d + t
                        heapq.heappush(pq, (d + t, *key))
                        updates.append(f"người {r} về:{t:.2f}")
        print('      cập nhật:', ', '.join(updates) if updates else '(không)')
    return -1.0

trace_dijkstra(3, 2, 3, [2, 5, 8], [1.0, 1.5, 0.75])

# 1 pop d=  0.00 | ở trại=111 giai đoạn=0 thuyền ở trại
      cập nhật: đi 110:8.00, đi 101:8.00, đi 100:8.00, đi 011:5.00, đi 010:5.00, đi 001:2.00
# 2 pop d=  2.00 | ở trại=110 giai đoạn=2 thuyền ở đích
      cập nhật: (không)
# 3 pop d=  5.00 | ở trại=100 giai đoạn=2 thuyền ở đích
      cập nhật: người 0 về:1.50, người 1 về:3.75
# 4 pop d=  5.00 | ở trại=101 giai đoạn=2 thuyền ở đích
      cập nhật: người 1 về:3.75
# 5 pop d=  6.50 | ở trại=101 giai đoạn=0 thuyền ở trại
      cập nhật: đi 101:8.00
# 6 pop d=  8.00 | ở trại=001 giai đoạn=2 thuyền ở đích
      cập nhật: người 1 về:3.75, người 2 về:6.00
# 7 pop d=  8.00 | ở trại=010 giai đoạn=2 thuyền ở đích
      cập nhật: người 0 về:1.50
# 8 pop d=  8.00 | ở trại=011 giai đoạn=2 thuyền ở đích
      cập nhật: (không)
# 9 pop d=  8.75 | ở trại=110 giai đoạn=2 thuyền ở trại
      cập nhật: (không)
#10 pop d=  8.75 | ở trại=111 giai đoạn=2 thuyền ở trại
      cập nhật: đi 001:1.50
#11 pop d=  9.50 | ở trại=011 giai đoạn=0 thuyền ở trại
 

14.5

Quan sát: Dijkstra mở rộng theo thứ tự tổng thời gian. Trạng thái đích được pop ở khoảng cách `14.5` — đúng đáp án. Nhiều trạng thái "đắt" (ví dụ đưa {1,2} sang trước) chưa bao giờ được pop.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def minTime(self, n: int, k: int, m: int, time: List[int], mul: List[float]) -> float:
        full = (1 << n) - 1
        mx = [0] * (1 << n)
        pc = [0] * (1 << n)
        for s in range(1, 1 << n):
            lb = s & -s
            mx[s] = max(mx[s ^ lb], time[lb.bit_length() - 1])
            pc[s] = pc[s ^ lb] + 1

        INF = float('inf')
        dist = {(full, 0, 0): 0.0}
        pq = [(0.0, full, 0, 0)]
        while pq:
            d, mask, st, side = heapq.heappop(pq)
            if d > dist[(mask, st, side)]:
                continue
            if side == 1 and mask == 0:
                return d
            if side == 0:
                sub = mask
                while sub:
                    if pc[sub] <= k:
                        t = mx[sub] * mul[st]
                        key = (mask ^ sub, (st + int(t) % m) % m, 1)
                        nd = d + t
                        if nd < dist.get(key, INF):
                            dist[key] = nd
                            heapq.heappush(pq, (nd, key[0], key[1], 1))
                    sub = (sub - 1) & mask
            else:
                dest = full ^ mask
                while dest:
                    lb = dest & -dest
                    dest ^= lb
                    t = time[lb.bit_length() - 1] * mul[st]
                    key = (mask | lb, (st + int(t) % m) % m, 0)
                    nd = d + t
                    if nd < dist.get(key, INF):
                        dist[key] = nd
                        heapq.heappush(pq, (nd, key[0], key[1], 0))
        return -1.0

for n, k, m, t, mul, exp in examples:
    got = Solution().minTime(n, k, m, t, mul)
    print(got, 'OK' if isclose(got, exp, abs_tol=1e-5) else 'SAI')

5.0 OK
14.5 OK
-1.0 OK


Ghi chú: thời gian luôn dương nên `int(t) == floor(t)`; dùng `int` nhanh hơn chút.

## 10. Vì sao thuật toán đúng

1. **Trạng thái đủ thông tin (tính Markov).** Chuyến đi tiếp theo chỉ phụ thuộc: ai ở bờ nào (`mask` + `side` của thuyền) và giai đoạn hiện tại. Tổng thời gian đã qua chỉ cộng dồn, không ảnh hưởng chi phí tương lai ⇒ bài toán là đường đi ngắn nhất trên đồ thị trạng thái.
2. **Mọi kế hoạch hợp lệ ↔ một đường đi** từ `(full, 0, 0)` đến `(0, *, 1)` và ngược lại; tổng thời gian = tổng trọng số cạnh.
3. **Trọng số dương** (`≥ 0.5`) ⇒ Dijkstra đúng: khi đỉnh được pop lần đầu, khoảng cách của nó tối ưu (mọi đường khác đi qua đỉnh còn trong heap có độ dài ≥ khoảng cách hiện tại).
4. **Dừng sớm đúng:** đỉnh đích đầu tiên được pop có khoảng cách nhỏ nhất trong mọi đỉnh đích (vì nó nhỏ nhất trong heap).
5. **`-1`:** heap cạn mà chưa chạm đích ⇔ không có đường ⇔ (`k = 1` và `n ≥ 2`).

## 11. Phân tích Time / Space Complexity

Gọi `N = 2^n`, `S = Σ_{mask} #{sub ⊆ mask : 1 ≤ |sub| ≤ k}` (≤ `3^n`; với `n = 12, k = 5` khoảng `4,3·10^5`, xem bài tập 3).

| Cách | Time | Space |
|---|---|---|
| 1. DFS đường đơn | mũ theo số trạng thái | `O(V)` |
| 2. Bellman-Ford | `O(V · E)` với `E ≈ m(S + nN)` | `O(E)` |
| 3. Dijkstra gộp | `O(m · S · n · log V)` | `O(m N)` |
| 4. Dijkstra hai phía | `O(m (S + nN) log V)` | `O(m N)` |

Với `n = 12, m = 5`: `E` vài triệu cạnh trong trường hợp xấu nhất; cách 4 thường dừng sớm và chạy dưới 1–2 giây trong Python.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
def close(a, b):
    return isclose(a, b, rel_tol=1e-9, abs_tol=1e-6)

impls = {
    'bellman': lambda *a: min_time_bellman(*a)[0],
    'merged': min_time_merged,
    'dijkstra': min_time_dijkstra,
    'submit': lambda *a: Solution().minTime(*a),
}
edge = [
    (1, 5, 1, [7], [0.5], 3.5),                  # một người, chuyến đi duy nhất
    (3, 5, 2, [1, 2, 3], [2.0, 1.0], 6.0),       # thuyền chở hết trong một chuyến
    (4, 1, 3, [1, 1, 1, 1], [1.0, 1.0, 1.0], -1.0),  # k = 1, n > 1
    (2, 2, 1, [100, 1], [2.0], 200.0),
]
for n, k, m, t, mul, exp in examples + edge:
    for name, f in impls.items():
        got = f(n, k, m, t, mul)
        assert close(got, exp), (name, n, k, m, t, mul, got, exp)
    assert close(min_time_brute(n, k, m, t, mul), exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên (seed cố định)

Vét cạn DFS chỉ chạy nổi với `n ≤ 3`; Bellman-Ford dùng làm oracle thứ hai cho `n ≤ 6`.

In [15]:
import random
random.seed(3594)
c1 = c2 = 0
for _ in range(120):
    n = random.randint(1, 3)
    k = random.randint(1, 3)
    m = random.randint(1, 3)
    t = [random.randint(1, 10) for _ in range(n)]
    mul = [random.choice([0.5, 0.75, 1.0, 1.25, 1.5, 2.0]) for _ in range(m)]
    exp = min_time_brute(n, k, m, t, mul)
    for name, f in impls.items():
        assert close(f(n, k, m, t, mul), exp), (name, n, k, m, t, mul)
    c1 += 1
for _ in range(80):
    n = random.randint(2, 6)
    k = random.randint(1, 4)
    m = random.randint(1, 5)
    t = [random.randint(1, 100) for _ in range(n)]
    mul = [round(random.uniform(0.5, 2.0), 2) for _ in range(m)]
    exp = min_time_bellman(n, k, m, t, mul)[0]
    for name in ['merged', 'dijkstra', 'submit']:
        assert close(impls[name](n, k, m, t, mul), exp), (name, n, k, m, t, mul)
    c2 += 1
print('Khớp vét cạn DFS:', c1, 'bộ; khớp Bellman-Ford:', c2, 'bộ.')

Khớp vét cạn DFS: 120 bộ; khớp Bellman-Ford: 80 bộ.


### 12.3. Ca lớn (`n = 12`, `k` từ 2 đến 5, `m = 5`)

In [16]:
import time as _time
random.seed(12)
for k in [2, 3, 4, 5]:
    t = [random.randint(1, 100) for _ in range(12)]
    mul = [round(random.uniform(0.5, 2.0), 2) for _ in range(5)]
    s = _time.time()
    ans = Solution().minTime(12, k, 5, t, mul)
    print(f"k={k}: đáp án {ans:.5f} | {_time.time() - s:.2f}s")

k=2: đáp án 359.01000 | 0.51s


k=3: đáp án 175.13000 | 0.63s


k=4: đáp án 161.93000 | 0.92s


k=5: đáp án 192.89000 | 0.88s


Mỗi ca dưới khoảng 1–2 giây trong Python. Cách 3 (gộp) chậm hơn rõ rệt ở `n = 12` vì số cạnh nhân lên `n` lần.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng DP/đệ quy có nhớ theo `(mask, stage)` như thể không có chu trình

In [17]:
import sys
from functools import lru_cache

def wrong_memo(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    visiting = set()

    @lru_cache(maxsize=None)
    def f(mask, st):
        # SAI: khi gặp lại trạng thái đang duyệt, coi như "vô cực" rồi lưu cache luôn
        if (mask, st) in visiting:
            return float('inf')
        visiting.add((mask, st))
        best = float('inf')
        sub = mask
        while sub:
            if pc[sub] <= k:
                t1 = mx[sub] * mul[st]
                st1 = (st + floor(t1) % m) % m
                rest = mask ^ sub
                if rest == 0:
                    best = min(best, t1)
                else:
                    for r in range(n):
                        if not rest >> r & 1:
                            t2 = time[r] * mul[st1]
                            best = min(best, t1 + t2 + f(rest | 1 << r, (st1 + floor(t2) % m) % m))
            sub = (sub - 1) & mask
        visiting.discard((mask, st))
        return best

    r = f(full, 0)
    return r if r < float('inf') else -1.0

random.seed(77)
found = None
for _ in range(400):
    n = random.randint(2, 4); k = random.randint(2, 3); m = random.randint(2, 4)
    t = [random.randint(1, 9) for _ in range(n)]
    mul = [random.choice([0.5, 1.0, 1.5, 2.0]) for _ in range(m)]
    a, b = wrong_memo(n, k, m, t, mul), Solution().minTime(n, k, m, t, mul)
    if not close(a, b):
        found = (n, k, m, t, mul, a, b)
        break
print('Phản ví dụ:', found)

Phản ví dụ: (3, 2, 4, [3, 9, 2], [2.0, 2.0, 1.5, 0.5], 24.0, 23.5)


**Vì sao sai:** trạng thái có chu trình; giá trị "vô cực" tạm thời khi gặp lại trạng thái đang duyệt bị **lưu vào cache** và dùng cho các nhánh khác ⇒ kết quả sai (thường lớn hơn đáp án). Đồ thị có chu trình cần Dijkstra (hoặc Bellman-Ford).

### Lỗi 2 — Luôn cho **người nhanh nhất** ở bờ đích quay về

In [18]:
def wrong_fastest_return(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    INF = float('inf')
    dist = {(full, 0, 0): 0.0}
    pq = [(0.0, full, 0, 0)]
    while pq:
        d, mask, st, side = heapq.heappop(pq)
        if d > dist[(mask, st, side)]:
            continue
        if side == 1 and mask == 0:
            return d
        if side == 0:
            sub = mask
            while sub:
                if pc[sub] <= k:
                    t = mx[sub] * mul[st]
                    key = (mask ^ sub, (st + floor(t) % m) % m, 1)
                    if d + t < dist.get(key, INF):
                        dist[key] = d + t
                        heapq.heappush(pq, (d + t, *key))
                sub = (sub - 1) & mask
        else:
            dest = [r for r in range(n) if not mask >> r & 1]
            r = min(dest, key=lambda i: time[i])          # SAI: chỉ thử người nhanh nhất
            t = time[r] * mul[st]
            key = (mask | 1 << r, (st + floor(t) % m) % m, 0)
            if d + t < dist.get(key, INF):
                dist[key] = d + t
                heapq.heappush(pq, (d + t, *key))
    return -1.0

random.seed(5)
found = None
for _ in range(3000):
    n = random.randint(3, 4); k = 2; m = random.randint(2, 5)
    t = [random.randint(1, 20) for _ in range(n)]
    mul = [random.choice([0.5, 1.0, 1.5, 2.0]) for _ in range(m)]
    a, b = wrong_fastest_return(n, k, m, t, mul), Solution().minTime(n, k, m, t, mul)
    if not close(a, b):
        found = (n, k, m, t, mul, a, b)
        break
print('Phản ví dụ (sai, đúng ở 2 số cuối):', found)

Phản ví dụ (sai, đúng ở 2 số cuối): (4, 2, 3, [13, 18, 4, 19], [1.0, 0.5, 1.0], 44.5, 44.0)


**Vì sao sai:** người chậm hơn quay về có thể làm giai đoạn nhảy tới một `mul` rất nhỏ, khiến chuyến sau rẻ hơn hẳn. Phải thử **mọi** người quay về.

### Lỗi 3 — Quên `% m` khi cập nhật giai đoạn

In [19]:
def stage_wrong(st, d, m):
    return st + floor(d) % m          # SAI: thiếu % m bên ngoài

def stage_right(st, d, m):
    return (st + floor(d) % m) % m

print('sai :', stage_wrong(2, 8.0, 3), '-> mul[4] sẽ IndexError khi m = 3')
print('đúng:', stage_right(2, 8.0, 3))

sai : 4 -> mul[4] sẽ IndexError khi m = 3
đúng: 1


**Vì sao sai:** giai đoạn là chỉ số vòng tròn `[0, m)`; thiếu `% m` sẽ truy cập ngoài mảng `mul` (hoặc sai chỉ số).

### Lỗi 4 — Bắt buộc mọi chuyến phải chở **đúng** `k` người

In [20]:
def wrong_exact_k(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    INF = float('inf')
    dist = {(full, 0, 0): 0.0}
    pq = [(0.0, full, 0, 0)]
    while pq:
        d, mask, st, side = heapq.heappop(pq)
        if d > dist[(mask, st, side)]:
            continue
        if side == 1 and mask == 0:
            return d
        if side == 0:
            sub = mask
            while sub:
                if pc[sub] == min(k, pc[mask]):              # SAI
                    t = mx[sub] * mul[st]
                    key = (mask ^ sub, (st + floor(t) % m) % m, 1)
                    if d + t < dist.get(key, INF):
                        dist[key] = d + t
                        heapq.heappush(pq, (d + t, *key))
                sub = (sub - 1) & mask
        else:
            for r in range(n):
                if not mask >> r & 1:
                    t = time[r] * mul[st]
                    key = (mask | 1 << r, (st + floor(t) % m) % m, 0)
                    if d + t < dist.get(key, INF):
                        dist[key] = d + t
                        heapq.heappush(pq, (d + t, *key))
    return -1.0

random.seed(9)
found = None
for _ in range(3000):
    n = random.randint(3, 5); k = random.randint(2, 3); m = random.randint(2, 5)
    t = [random.randint(1, 30) for _ in range(n)]
    mul = [random.choice([0.5, 1.0, 1.5, 2.0]) for _ in range(m)]
    a, b = wrong_exact_k(n, k, m, t, mul), Solution().minTime(n, k, m, t, mul)
    if not close(a, b):
        found = (n, k, m, t, mul, a, b)
        break
print('Phản ví dụ (sai, đúng ở 2 số cuối):', found)

Phản ví dụ (sai, đúng ở 2 số cuối): (4, 2, 5, [6, 23, 15, 24], [2.0, 1.0, 1.0, 1.0, 0.5], 90.5, 80.0)


**Vì sao sai:** "tối đa k người" — đôi khi chở ít người hơn lại tốt (ví dụ để giai đoạn rơi vào `mul` nhỏ, hoặc tránh ghép người chậm với nhóm).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 12–20` + "tập người/vật" ⇒ **bitmask**.
- Trạng thái có thể **quay lại** (người đi rồi về) ⇒ không làm DP thẳng; trọng số dương ⇒ **Dijkstra trên trạng thái**.
- Chuyển trạng thái "chọn một nhóm" ⇒ duyệt submask `(sub - 1) & mask`, tiền xử lý thuộc tính nhóm (`max`, `sum`, `popcount`) theo bit thấp nhất.
- Tách bước phức hợp (đi + về) thành **hai cạnh** với thêm chiều `side` ⇒ giảm số cạnh từ tích sang tổng.
- Kết quả số thực: so sánh với sai số; tính `floor` đúng như đề (nhân trực tiếp).

## 15. Biến thể của bài

1. Bài cổ điển "Bridge and Torch" (qua cầu với đèn pin, thuyền chở 2): đặc biệt khi `m = 1`, `mul = [1]` ⇒ có lời giải tham lam nổi tiếng.
2. Thời gian chuyến đi là **tổng** thay vì **max** ⇒ chỉ đổi bảng tiền xử lý.
3. Giai đoạn đổi theo **thời gian thực** (thời điểm hiện tại mod chu kỳ) ⇒ trạng thái cần thêm thời gian, phức tạp hơn.
4. Bài cùng pattern Dijkstra-trên-bitmask: LC 847 *Shortest Path Visiting All Nodes*, LC 864 *Shortest Path to Get All Keys*.

In [21]:
# Biến thể 1: m = 1, mul = [1.0], k = 2 — so với công thức tham lam "Bridge and Torch"
def torch_greedy(t):
    t = sorted(t)
    total = 0
    while len(t) > 3:
        a, b, y, z = t[0], t[1], t[-2], t[-1]
        total += min(b + a + z + b, z + a + y + a)
        t = t[:-2]
    if len(t) == 3:
        total += t[0] + t[1] + t[2]
    elif len(t) == 2:
        total += t[1]
    else:
        total += t[0]
    return total

random.seed(1)
for _ in range(50):
    t = [random.randint(1, 20) for _ in range(random.randint(1, 7))]
    assert close(Solution().minTime(len(t), 2, 1, t, [1.0]), torch_greedy(t))
print('Với m = 1, k = 2: Dijkstra khớp công thức Bridge and Torch.', torch_greedy([1, 2, 5, 10]))

Với m = 1, k = 2: Dijkstra khớp công thức Bridge and Torch. 17


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | DFS mọi đường đơn | mũ | `O(V)` | Oracle `n ≤ 3` |
| 2 | Bellman-Ford | `O(V·E)` | `O(E)` | Oracle `n ≤ 6`, hiểu khái niệm nới lỏng |
| 3 | Dijkstra gộp đi+về | `O(m·S·n·log)` | `O(mN)` | Ít chiều trạng thái hơn |
| 4 | Dijkstra hai phía | `O(m(S+nN) log)` | `O(mN)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minTime(2, 2, 1, [3, 4], [1.5])` = ?

**Bài 2 (tính tay):** `n=3, k=2, m=1, time=[1,2,5], mul=[1.0]` — đáp án bao nhiêu? (Bridge and Torch thu nhỏ.)

**Bài 3 (điền TODO):** viết `count_submasks(n, k)` = tổng số cặp `(mask, sub)` với `1 ≤ |sub| ≤ k` để ước lượng số cạnh.

**Bài 4 (tự viết lại):** sửa lời giải cho biến thể "thời gian chuyến đi = **tổng** `time` của nhóm".

**Bài 5 (mini challenge):** khôi phục **kế hoạch** tối ưu (danh sách chuyến đi/về) bằng mảng cha trong Dijkstra.

In [22]:
def count_submasks_todo(n, k):
    # TODO: duyệt mọi mask, mọi sub ⊆ mask, đếm những sub có 1 <= popcount <= k
    raise NotImplementedError

def min_time_sum_todo(n, k, m, time, mul):
    # TODO: như lời giải chính nhưng thời gian nhóm = tổng time
    raise NotImplementedError

def plan_todo(n, k, m, time, mul):
    # TODO: lưu parent[state] = (state_trước, hành động) rồi lần ngược
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = 2 ≥ n`. (2) Một chuyến: `max(3,4) × 1.5`. (3) `6.0`.

**Bài 2.** (1) Đưa {1,2} sang (2), 1 về (1), đưa {1,5} sang (5). (2) Tổng 8. (3) Không có cách nào nhanh hơn — kiểm tra bằng code.

**Bài 3.** (1) Hai vòng lặp mask/sub. (2) Hoặc công thức `Σ_j C(n,j) Σ_{i=1..min(j,k)} C(j,i)`. (3) So hai cách.

**Bài 4.** (1) Đổi bảng `mx` thành bảng `sm` (tổng). (2) `sm[s] = sm[s ^ lb] + time[idx(lb)]`. (3) Phần Dijkstra giữ nguyên.

**Bài 5.** (1) Khi cập nhật `dist[key]`, lưu `parent[key] = (state, mô tả)`. (2) Khi pop đích, lần ngược tới nguồn. (3) Đảo ngược danh sách và mô phỏng lại để kiểm tra tổng thời gian.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
a1 = Solution().minTime(2, 2, 1, [3, 4], [1.5])
a2 = Solution().minTime(3, 2, 1, [1, 2, 5], [1.0])
assert close(a1, 6.0) and close(a2, 8.0)
assert close(min_time_brute(3, 2, 1, [1, 2, 5], [1.0]), 8.0)
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 6.0 | Bài 2: 8.0


In [24]:
# Bài 3
from math import comb
def count_submasks(n, k):
    pc = [bin(s).count('1') for s in range(1 << n)]
    total = 0
    for mask in range(1 << n):
        sub = mask
        while sub:
            if pc[sub] <= k:
                total += 1
            sub = (sub - 1) & mask
    return total

def count_formula(n, k):
    return sum(comb(n, j) * sum(comb(j, i) for i in range(1, min(j, k) + 1)) for j in range(n + 1))

for n, k in [(3, 2), (8, 3), (12, 5)]:
    assert count_submasks(n, k) == count_formula(n, k)
print('n=12, k=5:', count_formula(12, 5), 'cặp (mask, sub)')

n=12, k=5: 432896 cặp (mask, sub)


In [25]:
# Bài 4: thời gian nhóm = tổng
def min_time_sum(n, k, m, time, mul):
    full = (1 << n) - 1
    sm = [0] * (1 << n)
    pc = [0] * (1 << n)
    for s in range(1, 1 << n):
        lb = s & -s
        sm[s] = sm[s ^ lb] + time[lb.bit_length() - 1]
        pc[s] = pc[s ^ lb] + 1
    INF = float('inf')
    dist = {(full, 0, 0): 0.0}
    pq = [(0.0, full, 0, 0)]
    while pq:
        d, mask, st, side = heapq.heappop(pq)
        if d > dist[(mask, st, side)]:
            continue
        if side == 1 and mask == 0:
            return d
        nxt = []
        if side == 0:
            sub = mask
            while sub:
                if pc[sub] <= k:
                    t = sm[sub] * mul[st]
                    nxt.append((t, (mask ^ sub, (st + floor(t) % m) % m, 1)))
                sub = (sub - 1) & mask
        else:
            for r in range(n):
                if not mask >> r & 1:
                    t = time[r] * mul[st]
                    nxt.append((t, (mask | 1 << r, (st + floor(t) % m) % m, 0)))
        for t, key in nxt:
            if d + t < dist.get(key, INF):
                dist[key] = d + t
                heapq.heappush(pq, (d + t, *key))
    return -1.0

print('Bài 4 ví dụ:', min_time_sum(3, 2, 3, [2, 5, 8], [1.0, 1.5, 0.75]))
assert close(min_time_sum(1, 1, 1, [4], [2.0]), 8.0)
assert min_time_sum(2, 1, 1, [1, 1], [1.0]) == -1.0

Bài 4 ví dụ: 23.5


In [26]:
# Bài 5: khôi phục kế hoạch
def plan(n, k, m, time, mul):
    full = (1 << n) - 1
    mx, pc = precompute(time)
    INF = float('inf')
    src = (full, 0, 0)
    dist, parent = {src: 0.0}, {src: None}
    pq = [(0.0, *src)]
    while pq:
        d, mask, st, side = heapq.heappop(pq)
        cur = (mask, st, side)
        if d > dist[cur]:
            continue
        if side == 1 and mask == 0:
            steps = []
            while parent[cur] is not None:
                cur, act = parent[cur]
                steps.append(act)
            return d, steps[::-1]
        moves = []
        if side == 0:
            sub = mask
            while sub:
                if pc[sub] <= k:
                    t = mx[sub] * mul[st]
                    moves.append((t, (mask ^ sub, (st + floor(t) % m) % m, 1),
                                  ('go', [i for i in range(n) if sub >> i & 1])))
                sub = (sub - 1) & mask
        else:
            for r in range(n):
                if not mask >> r & 1:
                    t = time[r] * mul[st]
                    moves.append((t, (mask | 1 << r, (st + floor(t) % m) % m, 0), ('back', r)))
        for t, key, act in moves:
            if d + t < dist.get(key, INF):
                dist[key] = d + t
                parent[key] = (cur, act)
                heapq.heappush(pq, (d + t, *key))
    return -1.0, []

total, steps = plan(3, 2, 3, [2, 5, 8], [1.0, 1.5, 0.75])
print('Kế hoạch:', steps)
assert close(simulate([2, 5, 8], [1.0, 1.5, 0.75], 3, steps), total)

Kế hoạch: [('go', [0, 1]), ('back', 0), ('go', [0, 2])]
  go [0, 1]: 5.00 phút, giai đoạn 0 -> 2, tổng 5.00
back 0: 1.50 phút, giai đoạn 2 -> 0, tổng 6.50
  go [0, 2]: 8.00 phút, giai đoạn 0 -> 2, tổng 14.50


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
mx[s], pc[s] theo bit thấp nhất
dist[(full, 0, 0)] = 0; heap = [(0, full, 0, 0)]
pop (d, mask, st, side); bỏ nếu lỗi thời
side==1 and mask==0 -> return d
side==0: với sub ⊆ mask, pc[sub] <= k: t = mx[sub]*mul[st] -> (mask^sub, (st+floor(t)%m)%m, 1)
side==1: với r ở bờ đích:  t = time[r]*mul[st]            -> (mask|1<<r, (st+floor(t)%m)%m, 0)
heap cạn -> -1
```

**Checklist:**

- [ ] Trạng thái có cả `side` (thuyền ở đâu) và `stage`?
- [ ] Dùng Dijkstra (không DP thẳng vì có chu trình)?
- [ ] Thử **mọi** nhóm `1..k` người và **mọi** người quay về?
- [ ] Giai đoạn mới `(st + floor(t) % m) % m`?
- [ ] Trường hợp `k = 1, n ≥ 2` trả `-1`?
- [ ] So sánh số thực bằng sai số khi test?